# CNN Spectrogram Classifier Training

**Goal**: train a CNN to classify seismic event type from 3-component (Z, N, E) spectrogram images built by `07a_spectrogram_dataset_build.py` on the ISTerre cluster (that script does the SDS access + instrument response removal + spectrogram computation).

!!!! This notebook only trains, since the cluster has no GPU

**Drive folder expected** — the packed output of `07a_consolidate_for_colab.py`
(run on the cluster after 07a; packs the many small per-sample .npz files into a
few large archives, since Google Drive's FUSE mount handles directories with
tens of thousands of files very poorly):
```
MyDrive/colab_cnn_training_spectrogram/
    spectrograms_train.npz   <- packed images + labels for the train split
    spectrograms_val.npz
    spectrograms_test.npz
    image_list.csv           <- manifest, kept for reference/provenance
    freq_axis.npy             <- shared frequency axis [Hz]
    time_axis.npy             <- shared time axis [s]
```

**Runtime on GoogleColab**: Runtime > Change runtime type > T4 GPU

## Cell 1 — Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print('No GPU — go to Runtime > Change runtime type > T4 GPU')

## Cell 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

## Cell 3 — Configure paths & hyperparameters
**Edit this cell** to match the Drive folder

In [ ]:
import os

# -- Edit here -----------------------------------------------------------
DRIVE_BASE      = '/content/drive/MyDrive/colab_cnn_training_spectrogram'
SPEC_TRAIN_PATH = os.path.join(DRIVE_BASE, 'spectrograms_train.npz')
SPEC_VAL_PATH   = os.path.join(DRIVE_BASE, 'spectrograms_val.npz')
SPEC_TEST_PATH  = os.path.join(DRIVE_BASE, 'spectrograms_test.npz')
MANIFEST_CSV    = os.path.join(DRIVE_BASE, 'image_list.csv')
FREQ_AXIS_PATH  = os.path.join(DRIVE_BASE, 'freq_axis.npy')
TIME_AXIS_PATH  = os.path.join(DRIVE_BASE, 'time_axis.npy')
LOG_DIR         = os.path.join(DRIVE_BASE, 'log')

# Must match TARGET_CLASSES in 07a_spectrogram_dataset_build.py
CLASS_NAMES = ['earthquake', 'regional', 'rockslide', 'ice quake', 'noise']

EPOCHS        = 60
BATCH_SIZE    = 128
LEARNING_RATE = 1.5e-4
DROPOUT_RATE  = 0.5

# Only applies to MODEL_BACKBONE == 'custom' 
USE_BLOCK_DROPOUT  = False     # adds a light Dropout(BLOCK_DROPOUT_RATE) after each of the 3 conv blocks (cell 9)
BLOCK_DROPOUT_RATE = 0.15      # dropout right before the final Dense layer

# Label smoothing caps the target probability at (1 - LABEL_SMOOTHING) instead of 1.0, which bounds how large any single prediction's loss contribution can get
#  -> 0 disables it
LABEL_SMOOTHING = 0.1

# Which optimizer Cell 9 compiles the model with:
#   'adam'    -- current default, used in every run so far
#   'nadam'   -- Adam + Nesterov momentum; usually a mild, low-risk upgrade over Adam
#   'adamw'   -- Adam with decoupled weight decay (proper L2, not folded into the gradient update) 
#   'rmsprop' -- adaptive per-parameter LR via a decaying average of squared gradients
#   'adagrad' -- accumulates ALL past squared gradients -> LR decays monotonically
#   'adamax'  -- Adam variant using the L-infinity norm instead of L2; sometimes more stable under large gradient outliers
#   'sgd'     -- classic SGD with momentum=0.9 (not adaptive -- every parameter shares one global LR)
OPTIMIZER = 'adamw'

# Decoupled weight decay, applied by the optimizer itself in cell 9's build_optimizer()
WEIGHT_DECAY = 0.004

# Toggle for Cell 8/10: how the training set handles class imbalance
# False: one shuffled pass per epoch over the natural class proportions (5-class train split: EQ 65.8%, noise 15.5%, RS 11.7%, regional 4.7%, IQ 2.3%)
#        -> imbalance corrected only via class_weight='balanced' in the loss
# True: per-class oversampling in Cell 8's train_ds
#        -> each class is drawn from its own shuffled+repeated pool at the proportions in CLASS_SAMPLE_WEIGHTS below
#        -> IQ is physically seen more often per epoch
USE_CLASS_OVERSAMPLING = True
CLASS_SAMPLE_WEIGHTS = {
    'earthquake': 0.45,
    'rockslide':  0.15,
    'ice quake':  0.12,
    'noise':      0.15,
    'regional':   0.13,
}

# 'custom'          -- the small from-scratch CNN in Cell 9 (build_cnn)
# 'resnet50'        -- ImageNet-pretrained ResNet50 backbone, fine-tuned on the spectrograms (build_resnet50_transfer, Cell 9)
# 'xception'        -- same tf.keras.applications topologies as resnet50, but weights=None
# 'vgg16', 'vgg19'        -> trained FROM SCRATCH on our spectrograms, not ImageNet-fine-tuned 
# 'densenet121'
# 'mobilenetv2'
MODEL_BACKBONE = 'custom'

# Only used when MODEL_BACKBONE == 'resnet50'
FINE_TUNE_LEARNING_RATE = 1e-4
RESNET_UNFREEZE_LAST_N = 15      # how many of ResNet50's final layers (out of 175 total) are left trainable
# --------------------------------------------------------------------------

os.makedirs(LOG_DIR, exist_ok=True)
for label, path in [('spectrograms_train.npz', SPEC_TRAIN_PATH),
                    ('spectrograms_val.npz',   SPEC_VAL_PATH),
                    ('spectrograms_test.npz',  SPEC_TEST_PATH),
                    ('image_list.csv',         MANIFEST_CSV),
                    ('freq_axis.npy',          FREQ_AXIS_PATH),
                    ('time_axis.npy',          TIME_AXIS_PATH)]:
    exists = os.path.exists(path)
    size_mb = f'  ({os.path.getsize(path) / 1e6:.1f} MB)' if exists else ''
    print(f"{'OK' if exists else 'MISSING'}  {label:24s}  {path}{size_mb}")

## Cell 3b — Set up a persistent run log

In [ ]:
import sys
import datetime

class _Tee:
    """Writes everything unmodified to `live_stream`"""
    def __init__(self, live_stream, log_stream):
        self.live_stream = live_stream
        self.log_stream = log_stream
        self._line_buf = ''

    def write(self, data):
        self.live_stream.write(data)
        for ch in data:
            if ch == '\r':
                self._line_buf = ''          # bar is about to redraw this line -- drop what we had
            elif ch == '\n':
                if self._line_buf.strip():
                    self.log_stream.write(self._line_buf + '\n')
                self._line_buf = ''
            else:
                self._line_buf += ch

    def flush(self):
        self.live_stream.flush()
        self.log_stream.flush()

RUN_LOG_PATH = os.path.join(LOG_DIR, f"run_log_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt")
_log_file = open(RUN_LOG_PATH, 'w', encoding='utf-8')
sys.stdout = _Tee(sys.__stdout__, _log_file)   # sys.__stdout__ (not sys.stdout) so reruns don't nest tees

print(f"{'='*70}")
print(f"  07b training run -- {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*70}")
print("  Cell 3 configuration:")
print(f"    CLASS_NAMES             = {CLASS_NAMES}")
print(f"    EPOCHS                  = {EPOCHS}")
print(f"    BATCH_SIZE              = {BATCH_SIZE}")
print(f"    LEARNING_RATE           = {LEARNING_RATE}")
print(f"    DROPOUT_RATE            = {DROPOUT_RATE}")
print(f"    USE_BLOCK_DROPOUT       = {USE_BLOCK_DROPOUT}")
print(f"    BLOCK_DROPOUT_RATE      = {BLOCK_DROPOUT_RATE}")
print(f"    LABEL_SMOOTHING         = {LABEL_SMOOTHING}")
print(f"    OPTIMIZER               = {OPTIMIZER!r}")
print(f"    WEIGHT_DECAY            = {WEIGHT_DECAY}")
print(f"    USE_CLASS_OVERSAMPLING  = {USE_CLASS_OVERSAMPLING}")
print(f"    CLASS_SAMPLE_WEIGHTS    = {CLASS_SAMPLE_WEIGHTS}")
print(f"    MODEL_BACKBONE          = {MODEL_BACKBONE!r}")
print(f"    FINE_TUNE_LEARNING_RATE = {FINE_TUNE_LEARNING_RATE}")
print(f"    RESNET_UNFREEZE_LAST_N  = {RESNET_UNFREEZE_LAST_N}")
print(f"{'='*70}")
print(f"\nLogging everything printed from here on -> {RUN_LOG_PATH}\n")


## Cell 4 — Check TensorFlow + GPU visibility

In [ ]:
import tensorflow as tf
print(f'TensorFlow : {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs       : {gpus}')
if not gpus:
    print('WARNING: no GPU — go to Runtime > Change runtime type > T4 GPU')

## Cell 5 — Load manifest and inspect class / split distribution

Splits (train/val/test) were already assigned by 07a on the cluster, by EVENT
(stratified by class). This notebook just reads the `split` column, no re-splitting.

In [ ]:
import pandas as pd
import numpy as np

manifest = pd.read_csv(MANIFEST_CSV)
print(f'Manifest: {len(manifest):,} rows')

label2idx = {name: i for i, name in enumerate(CLASS_NAMES)}
idx2label = {i: name for name, i in label2idx.items()}
print('Label encoding:', label2idx)

print('\nRows per split x class:')
print(manifest.groupby(['split', 'event_type']).size().unstack(fill_value=0).to_string())

freq_axis = np.load(FREQ_AXIS_PATH)
time_axis = np.load(TIME_AXIS_PATH)
print(f'\nFrequency axis: {len(freq_axis)} bins, 0-{freq_axis.max():.1f} Hz')
print(f'Time axis     : {len(time_axis)} bins, 0-{time_axis.max():.1f} s')

## Cell 6 — Load packed spectrogram archives into memory

Each split (train/val/test) is one consolidated `.npz` file built by `07a_consolidate_for_colab.py` — a single `np.load()` per split instead of tens of thousands of individual file reads.

Arrays are kept in **float16** (the dtype they were packed in) rather than upcast to float32 here — upcasting the whole dataset immediately doubles its RAM footprint. The model still trains in float32; the cast happens per-batch inside the tf.data pipeline in Cell 8.

**Only train and val are loaded here.** `X_test` is loaded lazily in Cell 13 instead, right before it's needed.

In [ ]:
import shutil

LOCAL_CACHE_DIR = '/content/spectrogram_cache'
os.makedirs(LOCAL_CACHE_DIR, exist_ok=True)

def to_local(drive_path):
    local_path = os.path.join(LOCAL_CACHE_DIR, os.path.basename(drive_path))
    if os.path.exists(local_path) and os.path.getsize(local_path) == os.path.getsize(drive_path):
        print(f'[cached] {local_path}')
        return local_path
    print(f'Copying {drive_path} -> {local_path} ...')
    shutil.copyfile(drive_path, local_path)
    print(f'[done] {local_path}  ({os.path.getsize(local_path) / 1e9:.2f} GB)')
    return local_path


def load_packed_split(path):
    local_path = to_local(path)
    with np.load(local_path, allow_pickle=False) as d:
        X = d['images']   # float16, as packed by 07a_consolidate_for_colab.py -- NOT upcast here
        y = np.array([label2idx[lbl] for lbl in d['labels']], dtype='int32')
    return X, y

X_train, y_train = load_packed_split(SPEC_TRAIN_PATH)
X_val,   y_val   = load_packed_split(SPEC_VAL_PATH)

print(f'Train: X={X_train.shape}  y={y_train.shape}  dtype={X_train.dtype}  ({X_train.nbytes / 1e9:.2f} GB)')
print(f'Val  : X={X_val.shape}  y={y_val.shape}  dtype={X_val.dtype}  ({X_val.nbytes / 1e9:.2f} GB)')

INPUT_SHAPE = X_train.shape[1:]
print(f'\nCNN input shape: {INPUT_SHAPE}')


## Cell 6b — Sanity-check the packed data (diagnostic)

This cell checks three things directly on the raw arrays, before Cell 7 normalizes them in place:

1. Are any images all-zero placeholders? (`07a_consolidate_for_colab.py` zero-fills a sample if its `.npz` failed to load)
2. Do per-class raw pixel statistics (mean/std/min/max) actually differ at all?
3. Does the labels array line up with the manifest's row order for each split?

In [ ]:
import gc

def chunked_stats(X, idx, chunk=1024):
    """Mean/std/min/max over X[idx], processed in small chunks so peak memory stays bounded regardless of how large the class is (no full-subset copy)"""
    total = 0.0
    total_sq = 0.0
    vmin, vmax = np.inf, -np.inf
    count = 0
    for start in range(0, len(idx), chunk):
        block = X[idx[start:start + chunk]].astype('float32')
        total += float(block.sum())
        total_sq += float((block ** 2).sum())
        vmin = min(vmin, float(block.min()))
        vmax = max(vmax, float(block.max()))
        count += block.size
        del block
    mean = total / count
    var = max(total_sq / count - mean ** 2, 0.0)
    return mean, var ** 0.5, vmin, vmax

def check_split(name, X, y, manifest_split):
    n = len(y)
    n_zero = 0
    for start in range(0, n, 1024):
        block = X[start:start + 1024]
        n_zero += int((~block.reshape(len(block), -1).any(axis=1)).sum())
        del block
    print(f'--- {name} ---')
    print(f'  Zero-filled images (failed loads in consolidate step): {n_zero}/{n} ({100*n_zero/n:.2f}%)')

    for cls_idx, cls_name in idx2label.items():
        idx = np.where(y == cls_idx)[0]
        if len(idx) == 0:
            continue
        mean, std, vmin, vmax = chunked_stats(X, idx)
        print(f'  {cls_name:12s} n={len(idx):6d}  mean={mean:8.3f}  std={std:8.3f}  '
              f'min={vmin:8.3f}  max={vmax:8.3f}')

    manifest_labels = manifest_split['event_type'].values
    loaded_labels = np.array([idx2label[i] for i in y])
    match = np.mean(manifest_labels[:n] == loaded_labels[:len(manifest_labels)])
    print(f'  Label order match vs manifest: {match*100:.1f}%  (should be ~100%)')
    gc.collect()

for split_name, X, y in [('train', X_train, y_train), ('val', X_val, y_val)]:
    manifest_split = manifest[manifest['split'] == split_name].reset_index(drop=True)
    check_split(split_name, X, y, manifest_split)

## Cell 7 — Per-channel normalization (fit on TRAIN only)

In [ ]:
# dtype='float32' forces the mean/std reduction to accumulate in float32 even though X_train is float16
channel_mean = X_train.mean(axis=(0, 1, 2), keepdims=True, dtype='float32')
channel_std  = X_train.std(axis=(0, 1, 2), keepdims=True, dtype='float32') + 1e-8

np.savez(os.path.join(LOG_DIR, 'normalization_stats.npz'),
         mean=channel_mean, std=channel_std)
print('Per-channel mean:', channel_mean.ravel())
print('Per-channel std :', channel_std.ravel())

# Normalize X_train / X_val IN PLACE, staying float16
mean16 = channel_mean.astype('float16')
std16  = channel_std.astype('float16')

X_train -= mean16
X_train /= std16
X_val   -= mean16
X_val   /= std16

X_train_n = X_train   
X_val_n   = X_val

print(f'\nSaved -> normalization_stats.npz')
print(f'X_train_n: dtype={X_train_n.dtype}  ({X_train_n.nbytes / 1e9:.2f} GB)')
print(f'X_val_n  : dtype={X_val_n.dtype}  ({X_val_n.nbytes / 1e9:.2f} GB)')

## Cell 8 — Data augmentation (SpecAugment-style) + tf.data pipeline

In [ ]:
def spec_augment(image, label, n_time_masks=1, n_freq_masks=1,
                 max_time_mask_frac=0.15, max_freq_mask_frac=0.15, noise_std=0.05):
    img = tf.identity(image)
    n_freq  = tf.shape(img)[0]
    n_time  = tf.shape(img)[1]

    for _ in range(n_time_masks):
        mask_w = tf.random.uniform([], 0, tf.cast(tf.cast(n_time, tf.float32) * max_time_mask_frac, tf.int32) + 1, dtype=tf.int32)
        mask_w = tf.maximum(mask_w, 1)
        t0 = tf.random.uniform([], 0, tf.maximum(n_time - mask_w, 1), dtype=tf.int32)
        time_idx = tf.range(n_time)
        time_mask = tf.logical_and(time_idx >= t0, time_idx < t0 + mask_w)
        time_mask = tf.cast(tf.logical_not(time_mask), img.dtype)[tf.newaxis, :, tf.newaxis]
        img = img * time_mask

    for _ in range(n_freq_masks):
        mask_h = tf.random.uniform([], 0, tf.cast(tf.cast(n_freq, tf.float32) * max_freq_mask_frac, tf.int32) + 1, dtype=tf.int32)
        mask_h = tf.maximum(mask_h, 1)
        f0 = tf.random.uniform([], 0, tf.maximum(n_freq - mask_h, 1), dtype=tf.int32)
        freq_idx = tf.range(n_freq)
        freq_mask = tf.logical_and(freq_idx >= f0, freq_idx < f0 + mask_h)
        freq_mask = tf.cast(tf.logical_not(freq_mask), img.dtype)[:, tf.newaxis, tf.newaxis]
        img = img * freq_mask

    img = img + tf.random.normal(tf.shape(img), mean=0.0, stddev=noise_std, dtype=img.dtype)
    return img, label

def to_float32(image, label):
    return tf.cast(image, tf.float32), label

AUTOTUNE = tf.data.AUTOTUNE

def make_lookup_from_indices(index_ds, X, y, img_dtype):
    """Given a dataset of integer indices into X/y, fetch (image, label) pairs via tf.numpy_function"""
    def _fetch(i):
        return X[i], y[i]

    def fetch_fn(i):
        img, lbl = tf.numpy_function(_fetch, [i], [img_dtype, tf.int32])
        img.set_shape(INPUT_SHAPE)
        lbl.set_shape([])
        return img, lbl

    return index_ds.map(fetch_fn, num_parallel_calls=AUTOTUNE)


def make_lookup_ds(X, y, img_dtype, shuffle_buffer=None, seed=None):
    n = len(y)
    ds = tf.data.Dataset.range(n)
    if shuffle_buffer:
        ds = ds.shuffle(buffer_size=shuffle_buffer, seed=seed)
    return make_lookup_from_indices(ds, X, y, img_dtype)


def make_oversampled_train_ds(X, y, img_dtype, class_sample_weights, seed=42):
    """Per-class oversampling"""
    weights = np.array([class_sample_weights[name] for name in CLASS_NAMES], dtype='float64')
    weights = weights / weights.sum()

    print('Per-class training sample weights (target batch composition):')
    for name, w in zip(CLASS_NAMES, weights):
        natural_pct = 100 * np.mean(y == label2idx[name])
        print(f'  {name:12s} target={w*100:5.1f}%   natural={natural_pct:5.1f}%')

    per_class_ds = []
    for class_idx, name in enumerate(CLASS_NAMES):
        idx = np.where(y == class_idx)[0]
        ds = tf.data.Dataset.from_tensor_slices(idx)
        ds = ds.shuffle(buffer_size=len(idx), seed=seed, reshuffle_each_iteration=True)
        ds = ds.repeat()
        per_class_ds.append(make_lookup_from_indices(ds, X, y, img_dtype))

    return tf.data.Dataset.sample_from_datasets(per_class_ds, weights=list(weights), seed=seed)

SHUFFLE_BUFFER = len(y_train)

if USE_CLASS_OVERSAMPLING:
    train_ds = (make_oversampled_train_ds(X_train_n, y_train, tf.float16, CLASS_SAMPLE_WEIGHTS, seed=42)
                .take(len(y_train))   # cap epoch length so it's comparable to the non-oversampled path
                .map(spec_augment, num_parallel_calls=AUTOTUNE)
                .map(to_float32, num_parallel_calls=AUTOTUNE)
                .batch(BATCH_SIZE)
                .prefetch(AUTOTUNE))
else:
    train_ds = (make_lookup_ds(X_train_n, y_train, tf.float16, shuffle_buffer=SHUFFLE_BUFFER, seed=42)
                .map(spec_augment, num_parallel_calls=AUTOTUNE)
                .map(to_float32, num_parallel_calls=AUTOTUNE)
                .batch(BATCH_SIZE)
                .prefetch(AUTOTUNE))
print(f'train_ds built with USE_CLASS_OVERSAMPLING={USE_CLASS_OVERSAMPLING}')

val_ds = (make_lookup_ds(X_val_n, y_val, tf.float16)
          .map(to_float32, num_parallel_calls=AUTOTUNE)
          .batch(BATCH_SIZE)
          .prefetch(AUTOTUNE))

# test_ds is built in Cell 13 instead, once X_test is loaded there
print('tf.data pipelines ready (train_ds, val_ds -- test_ds built later in Cell 13).')

## Cell 9 — Build the CNN

In [ ]:
from tensorflow.keras import layers, models

def build_cnn(input_shape, n_classes, dropout_rate=0.5,
              use_block_dropout=False, block_dropout_rate=0.15):
    inputs = layers.Input(shape=input_shape, name='spectrogram')

    x = inputs
    for i, filters in enumerate([32, 64, 128]):
        x = layers.Conv2D(filters, 3, padding='same', use_bias=False, name=f'conv{i+1}_a')(x)
        x = layers.BatchNormalization(name=f'bn{i+1}_a')(x)
        x = layers.Activation('relu', name=f'relu{i+1}_a')(x)
        x = layers.Conv2D(filters, 3, strides=2, padding='same', use_bias=False, name=f'conv{i+1}_down')(x)
        x = layers.BatchNormalization(name=f'bn{i+1}_down')(x)
        x = layers.Activation('relu', name=f'relu{i+1}_down')(x)
        # Light dropout after each block, on top of the existing dropout right before the final Dense
        if use_block_dropout:
            x = layers.Dropout(block_dropout_rate, name=f'block_dropout{i+1}')(x)

    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dropout(dropout_rate, name='dropout')(x)
    outputs = layers.Dense(n_classes, activation='softmax', name='predictions')(x)

    return models.Model(inputs, outputs, name='spectrogram_cnn')


def build_resnet50_transfer(input_shape, n_classes, dropout_rate=0.5, unfreeze_last_n=15):
    """ImageNet-pretrained ResNet50 backbone + new head, fine-tuned on our spectrograms"""
    base = tf.keras.applications.ResNet50(
        include_top=False,
        weights='imagenet',
        input_shape=input_shape,
        pooling='avg',
    )
    base.trainable = True
    for layer in base.layers[:-unfreeze_last_n]:
        layer.trainable = False
    n_trainable = sum(1 for l in base.layers if l.trainable)
    print(f'ResNet50 backbone: {n_trainable}/{len(base.layers)} layers trainable '
          f'(last {unfreeze_last_n} unfrozen)')

    x = layers.Dropout(dropout_rate, name='dropout')(base.output)
    outputs = layers.Dense(n_classes, activation='softmax', name='predictions')(x)

    return models.Model(base.input, outputs, name='spectrogram_resnet50_transfer')


SCRATCH_BACKBONE_CLASSES = {
    'xception':    tf.keras.applications.Xception,
    'vgg16':       tf.keras.applications.VGG16,
    'vgg19':       tf.keras.applications.VGG19,
    'densenet121': tf.keras.applications.DenseNet121,
    'mobilenetv2': tf.keras.applications.MobileNetV2,
}


def build_scratch_backbone(name, input_shape, n_classes, dropout_rate=0.5):
    """Same tf.keras.applications topology as build_resnet50_transfer, but weights=None"""
    if name not in SCRATCH_BACKBONE_CLASSES:
        raise ValueError(
            f"Unknown scratch backbone {name!r}, expected one of: {sorted(SCRATCH_BACKBONE_CLASSES)}"
        )
    backbone_cls = SCRATCH_BACKBONE_CLASSES[name]
    base = backbone_cls(
        include_top=False,
        weights=None,
        input_shape=input_shape,
        pooling='avg',
    )
    print(f'{name} backbone (from scratch): {base.count_params():,} params, '
          f'{len(base.layers)} layers, all trainable')

    x = layers.Dropout(dropout_rate, name='dropout')(base.output)
    outputs = layers.Dense(n_classes, activation='softmax', name='predictions')(x)

    return models.Model(base.input, outputs, name=f'spectrogram_{name}_scratch')


if MODEL_BACKBONE == 'resnet50':
    model = build_resnet50_transfer(INPUT_SHAPE, len(CLASS_NAMES), dropout_rate=DROPOUT_RATE,
                                     unfreeze_last_n=RESNET_UNFREEZE_LAST_N)
    active_lr = FINE_TUNE_LEARNING_RATE
elif MODEL_BACKBONE == 'custom':
    model = build_cnn(INPUT_SHAPE, len(CLASS_NAMES), dropout_rate=DROPOUT_RATE,
                       use_block_dropout=USE_BLOCK_DROPOUT, block_dropout_rate=BLOCK_DROPOUT_RATE)
    active_lr = LEARNING_RATE
elif MODEL_BACKBONE in SCRATCH_BACKBONE_CLASSES:
    model = build_scratch_backbone(MODEL_BACKBONE, INPUT_SHAPE, len(CLASS_NAMES), dropout_rate=DROPOUT_RATE)
    active_lr = LEARNING_RATE
else:
    raise ValueError(
        f"Unknown MODEL_BACKBONE={MODEL_BACKBONE!r}, expected 'custom', 'resnet50', "
        f"or one of {sorted(SCRATCH_BACKBONE_CLASSES)}"
    )

def make_smoothed_sparse_ce(label_smoothing):
    """Sparse-label-compatible cross-entropy with label smoothing"""
    def loss_fn(y_true, y_pred):
        y_true_int = tf.cast(y_true, tf.int32)
        n_classes = tf.shape(y_pred)[-1]
        y_true_one_hot = tf.one_hot(y_true_int, depth=n_classes)
        return tf.keras.losses.categorical_crossentropy(
            y_true_one_hot, y_pred, label_smoothing=label_smoothing,
        )
    return loss_fn


def build_optimizer(name, lr, weight_decay=None):
    """Name -> tf.keras.optimizers instance"""
    name = name.lower()
    if name == 'adam':
        return tf.keras.optimizers.Adam(learning_rate=lr, weight_decay=weight_decay)
    elif name == 'nadam':
        return tf.keras.optimizers.Nadam(learning_rate=lr, weight_decay=weight_decay)
    elif name == 'adamw':
        return tf.keras.optimizers.AdamW(learning_rate=lr, weight_decay=weight_decay)
    elif name == 'rmsprop':
        return tf.keras.optimizers.RMSprop(learning_rate=lr, weight_decay=weight_decay)
    elif name == 'adagrad':
        return tf.keras.optimizers.Adagrad(learning_rate=lr, weight_decay=weight_decay)
    elif name == 'adamax':
        return tf.keras.optimizers.Adamax(learning_rate=lr, weight_decay=weight_decay)
    elif name == 'sgd':
        return tf.keras.optimizers.SGD(learning_rate=lr, momentum=0.9, weight_decay=weight_decay)
    else:
        raise ValueError(
            f"Unknown OPTIMIZER={name!r}, expected one of: "
            f"'adam', 'nadam', 'adamw', 'rmsprop', 'adagrad', 'adamax', 'sgd'"
        )


print(f'MODEL_BACKBONE={MODEL_BACKBONE!r}  learning_rate={active_lr}  '
      f'label_smoothing={LABEL_SMOOTHING}  optimizer={OPTIMIZER!r}  weight_decay={WEIGHT_DECAY}')
if MODEL_BACKBONE == 'custom':
    print(f'use_block_dropout={USE_BLOCK_DROPOUT}  block_dropout_rate={BLOCK_DROPOUT_RATE}')
model.compile(
    optimizer=build_optimizer(OPTIMIZER, active_lr, WEIGHT_DECAY),
    loss=make_smoothed_sparse_ce(LABEL_SMOOTHING),
    metrics=['accuracy'],
)
model.summary()


## Cell 10 — Class weights + callbacks

SMOTE doesn't apply to images, so `class_weight='balanced'` (plus the augmentation in Cell 8) is how class imbalance gets handled here

When `USE_CLASS_OVERSAMPLING` (Cell 3) is `True`, `class_weight` is set to `None` here. Imbalance is already being corrected by oversampling in `train_ds` (Cell 8), so also reweighting the loss would double-correct for the same thing

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

if USE_CLASS_OVERSAMPLING:
    # Oversampling in train_ds (Cell 8) already handles class imbalance directly --
    # stacking class_weight on top would double-correct and risks reintroducing a gradient-variance instability
    class_weight_dict = None
    print('USE_CLASS_OVERSAMPLING=True -> class_weight=None (imbalance handled by train_ds sampling instead)')
else:
    class_weight_values = compute_class_weight('balanced', classes=np.arange(len(CLASS_NAMES)), y=y_train)
    class_weight_dict = {i: w for i, w in enumerate(class_weight_values)}
    print('Class weights:', {idx2label[i]: round(w, 3) for i, w in class_weight_dict.items()})

checkpoint_path = os.path.join(LOG_DIR, 'best_model.keras')
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(checkpoint_path, monitor='val_loss', save_best_only=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6),
]
print(f'Checkpoints -> {checkpoint_path}')

## Cell 11 — Train

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1,   
)

# Persist the full per-epoch history so plots/reports can be regenerated later without retraining
history_path = os.path.join(LOG_DIR, 'training_history.csv')
pd.DataFrame(history.history).to_csv(history_path, index_label='epoch')
print(f'\n[SAVED] {history_path}')

best_epoch = int(np.argmin(history.history['val_loss']))
print(f"Best epoch (lowest val_loss): {best_epoch + 1}  "
      f"(val_loss={history.history['val_loss'][best_epoch]:.4f}, "
      f"val_accuracy={history.history['val_accuracy'][best_epoch]:.4f})")
print("(EarlyStopping restore_best_weights=True -> this is the model now in memory.)")


## Cell 12 — Training curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(history.history['loss'], label='train')
axes[0].plot(history.history['val_loss'], label='val')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].set_title('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['accuracy'], label='train')
axes[1].plot(history.history['val_accuracy'], label='val')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy'); axes[1].set_title('Accuracy')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
curves_path = os.path.join(LOG_DIR, 'training_curves.png')
plt.savefig(curves_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'[SAVED] {curves_path}')

# Standalone loss-only plot
fig_loss, ax_loss = plt.subplots(figsize=(7, 5))
ax_loss.plot(history.history['loss'], label='train')
ax_loss.plot(history.history['val_loss'], label='val')
ax_loss.set_xlabel('Epoch'); ax_loss.set_ylabel('Loss')
ax_loss.set_title('Loss evolution over training')
ax_loss.legend(); ax_loss.grid(True, alpha=0.3)
plt.tight_layout()
loss_only_path = os.path.join(LOG_DIR, 'loss_curve.png')
plt.savefig(loss_only_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'[SAVED] {loss_only_path}')

## Cell 13 — Evaluate on test set

Classification report, confusion matrix, one-vs-rest ROC curves — so CNN and HGB results are directly comparable

`X_test`/`y_test` are loaded here so the test split only occupies RAM for evaluation + Grad-CAM (Cell 14), not for the whole training run

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
from sklearn.preprocessing import label_binarize

X_test, y_test = load_packed_split(SPEC_TEST_PATH)
print(f'Test : X={X_test.shape}  y={y_test.shape}  dtype={X_test.dtype}  ({X_test.nbytes / 1e9:.2f} GB)')

X_test_n = (X_test.astype('float32') - channel_mean) / channel_std

test_ds = (make_lookup_ds(X_test_n, y_test, tf.float32)
           .map(to_float32, num_parallel_calls=AUTOTUNE)
           .batch(BATCH_SIZE)
           .prefetch(AUTOTUNE))

y_proba = model.predict(test_ds)
y_pred  = np.argmax(y_proba, axis=1)

print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

cm_pct = confusion_matrix(y_test, y_pred, labels=np.arange(len(CLASS_NAMES)), normalize='true')
print('Confusion matrix (%, rows=true, cols=predicted -- each row sums to 100%):')
print((pd.DataFrame(cm_pct, index=CLASS_NAMES, columns=CLASS_NAMES) * 100).round(1).to_string())

fig_cm, ax_cm = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix=cm_pct, display_labels=CLASS_NAMES).plot(
    ax=ax_cm, cmap='Blues', colorbar=False, values_format='.1%')
ax_cm.set_title('Confusion matrix (%) — test set (CNN)')
plt.tight_layout()
cm_path = os.path.join(LOG_DIR, 'confusion_matrix.png')
plt.savefig(cm_path, dpi=150)
plt.show()
print(f'[SAVED] {cm_path}')

y_test_bin = label_binarize(y_test, classes=np.arange(len(CLASS_NAMES)))
fig_roc, ax_roc = plt.subplots(figsize=(7, 5))
for i, name in enumerate(CLASS_NAMES):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
    ax_roc.plot(fpr, tpr, lw=2, label=f'{name}  (AUC={auc(fpr, tpr):.3f})')
ax_roc.plot([0, 1], [0, 1], 'k--', lw=1)
ax_roc.set_xlabel('False Positive Rate'); ax_roc.set_ylabel('True Positive Rate')
ax_roc.set_title('ROC curves — one-vs-rest (test set, CNN)')
ax_roc.legend(loc='lower right')
plt.tight_layout()
roc_path = os.path.join(LOG_DIR, 'roc_curves.png')
plt.savefig(roc_path, dpi=150)
plt.show()
print(f'[SAVED] {roc_path}')

## Cell 14 — Grad-CAM on example test images

Shows which time-frequency region the CNN is keying on for a few correctly-classified examples per class

In [ ]:
last_conv_name = None
for layer in model.layers:
    if isinstance(layer, layers.Conv2D):
        last_conv_name = layer.name
print(f'Grad-CAM target layer: {last_conv_name}')

grad_model = tf.keras.models.Model(model.inputs, [model.get_layer(last_conv_name).output, model.output])

def grad_cam(img_batch, class_idx):
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_batch)
        loss = preds[:, class_idx]
    grads = tape.gradient(loss, conv_out)
    weights = tf.reduce_mean(grads, axis=(1, 2), keepdims=True)
    cam = tf.reduce_sum(weights * conv_out, axis=-1)
    cam = tf.nn.relu(cam)
    cam = cam / (tf.reduce_max(cam, axis=(1, 2), keepdims=True) + 1e-8)
    return cam.numpy()

fig, axes = plt.subplots(len(CLASS_NAMES), 2, figsize=(9, 3.2 * len(CLASS_NAMES)))
for row, name in enumerate(CLASS_NAMES):
    idx_candidates = np.where((y_test == label2idx[name]) & (y_pred == label2idx[name]))[0]
    if len(idx_candidates) == 0:
        for col in range(2):
            axes[row, col].text(0.5, 0.5, f'No correct\n{name} example', ha='center', va='center')
            axes[row, col].axis('off')
        continue
    sample_idx = idx_candidates[0]
    img_batch = X_test_n[sample_idx:sample_idx + 1]
    cam = grad_cam(img_batch, label2idx[name])[0]

    axes[row, 0].imshow(X_test[sample_idx][:, :, 0], aspect='auto', origin='lower', cmap='viridis')
    axes[row, 0].set_title(f'{name} — Z channel (raw dB)')
    axes[row, 1].imshow(X_test[sample_idx][:, :, 0], aspect='auto', origin='lower', cmap='gray')
    axes[row, 1].imshow(cam, aspect='auto', origin='lower', cmap='jet', alpha=0.45,
                        extent=axes[row, 1].get_xlim() + axes[row, 1].get_ylim())
    axes[row, 1].set_title(f'{name} — Grad-CAM overlay')

plt.tight_layout()
gradcam_path = os.path.join(LOG_DIR, 'grad_cam_examples.png')
plt.savefig(gradcam_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'[SAVED] {gradcam_path}')

## Cell 15 — Save final model

In [ ]:
final_model_path = os.path.join(LOG_DIR, 'spectrogram_cnn_final.keras')
model.save(final_model_path)
print(f'[SAVED] {final_model_path}')
print(f'\nReload with:')
print(f"  model = tf.keras.models.load_model('{final_model_path}')")
print(f'\nDon\'t forget: apply the SAME normalization at inference time —')
print(f"  stats = np.load('{os.path.join(LOG_DIR, 'normalization_stats.npz')}')")
print(f"  X_new_norm = (X_new - stats['mean']) / stats['std']")

## Cell 16 — Close the run log

In [ ]:
print(f"\n{'='*70}")
print(f"  Run log saved -> {RUN_LOG_PATH}")
print(f"{'='*70}")

sys.stdout = sys.__stdout__
_log_file.close()